# Practical 08: Columnar Storage with Apache Parquet Format in PySpark

**UNIT 2: Big Data Processing using PySpark**

**Problem Statement:** Write a PySpark program to demonstrate high-performance columnar storage using Apache Parquet. Create a PySpark DataFrame, export it to Parquet format using .write.mode('overwrite').parquet(), reload the Parquet file using spark.read.parquet(), and verify schema preservation and row counts.

In [1]:
import os
import shutil
import tempfile
from pyspark.sql import SparkSession, Row

spark = (SparkSession.builder.appName("DS505-Practical08").master("local[*]")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

data = [
    {"Name": "Ananya", "Department": "IT", "City": "Vadodara", "Salary": 69000, "Rating": 4.3},
    {"Name": "Yash", "Department": "HR", "City": "Surat", "Salary": 51000, "Rating": 3.8},
    {"Name": "Pooja", "Department": "Finance", "City": "Rajkot", "Salary": 63000, "Rating": 4.0},
    {"Name": "Harsh", "Department": "IT", "City": "Surat", "Salary": 85000, "Rating": 4.6},
    {"Name": "Simran", "Department": "Sales", "City": "Pune", "Salary": 46000, "Rating": 3.5},
]
df = spark.createDataFrame([Row(**d) for d in data])
df.printSchema()
df.show()

tmp_dir = tempfile.mkdtemp()
path = os.path.join(tmp_dir, "employees.parquet")

df.write.mode("overwrite").parquet(path)
print("Parquet written to:", path)
print("Files inside      :", [f for f in os.listdir(path) if not f.startswith(".")])

df_back = spark.read.parquet(path)
df_back.printSchema()
df_back.show()

print("Schema preserved :", df.schema == df_back.schema)
print(f"Row count original = {df.count()} | reloaded = {df_back.count()} | equal = {df.count() == df_back.count()}")

df_back.select("Salary").show()

shutil.rmtree(tmp_dir)
spark.stop()
print("Cleaned up.")

root
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Salary: long (nullable = true)
 |-- Rating: double (nullable = true)

+------+----------+--------+------+------+
|  Name|Department|    City|Salary|Rating|
+------+----------+--------+------+------+
|Ananya|        IT|Vadodara| 69000|   4.3|
|  Yash|        HR|   Surat| 51000|   3.8|
| Pooja|   Finance|  Rajkot| 63000|   4.0|
| Harsh|        IT|   Surat| 85000|   4.6|
|Simran|     Sales|    Pune| 46000|   3.5|
+------+----------+--------+------+------+

Parquet written to: /tmp/tmp2xvdta7e/employees.parquet
Files inside      : ['_SUCCESS', 'part-00000-679e83bb-daab-493d-a394-0af57a95ebdc-c000.snappy.parquet', 'part-00001-679e83bb-daab-493d-a394-0af57a95ebdc-c000.snappy.parquet']
root
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Salary: long (nullable = true)
 |-- Rating: double (nullable = t